# # Install dependencies

In [1]:
!pip install -q --user scikit-learn torch


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: Can not perform a '--user' install. User site-packages are not visible in this virtualenv.


# # Load training data

In [2]:
"""Load the training data.

Pick the form that matches your storage:

    df = data.connection(conn_name).load("SELECT * FROM iris")   # SQL Explorer connection
    df = data.load("/home/jovyan/data/iris.csv")                 # local file (csv/parquet)
    df = data.load("s3://my-bucket/iris.parquet")                # S3 object

`data.split` shuffles rows by default (use `shuffle=False` to keep order,
e.g. for time-series). `random_state` makes the split reproducible.

`prepare()` is demo scaffolding — replace with your own data source in production.
"""
from src.demo import prepare
conn_name = prepare()

from nubison_model import data

df = data.connection(conn_name).load("SELECT * FROM iris")
datasets = data.split(df, {"train": 0.6, "val": 0.2, "test": 0.2}, random_state=42)

for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: 

Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


  color_warning(


train  rows= 90

val    rows= 30

test   rows= 30

# # Train (PyTorch)

In [3]:
# `train(datasets, target, *, ...)` parameters:
#   datasets      — dict from `data.split` (must include "train";
#                   "val" enables `t.X_val / t.y_val` + auto-scoring;
#                   "test" populates `t.X_test / t.y_test`)
#   target        — label column name(s); single string or list of strings.
#                   Splits each split's DataFrame into X = features and
#                   y = target so estimators can call `fit(X, y)`.
#   model_type    — free-form string tagged on the run as `model_type`
#                   (surfaced in the nubison UI). "classifier" and
#                   "regressor" additionally make `t.fit()` log
#                   `val_accuracy` / `val_r2`. None skips the tag.
#   weights_path  — where `t.save(model)` writes the pickle.
#                   Default "src/weights.pkl" matches `register(artifact_dirs="src")`.
import torch
import torch.nn as nn
from nubison_model import train


class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))

    def forward(self, x):
        return self.fc(x)


with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    X = torch.tensor(t.X_train.values, dtype=torch.float32)
    y = torch.tensor(t.y_train.values.ravel(), dtype=torch.long)

    model = IrisNet()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(30):
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        t.log_metric("loss", loss.item(), step=epoch)

    t.save(model)

print(f"run_id: {t.run_id}")


2026/05/15 22:14:50 INFO mlflow.tracking.fluent: Experiment with name 'Shuf_1778850874_train_pytorch' does not exist. Creating a new experiment.


2026/05/15 22:14:51 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run traveling-hawk-27 at: http://127.0.0.1:5050/#/experiments/24/runs/db2cd3db93c84a80a2078bc9a48fb303


🧪 View experiment at: http://127.0.0.1:5050/#/experiments/24


run_id: db2cd3db93c84a80a2078bc9a48fb303